# B2S 06 - AndinaLog Warehouse Costs

Conversión Bronze a Silver de costos de almacén por centro de distribución y periodo calendario. La fuente Bronze se conserva sin modificaciones; las transformaciones y decisiones de enrutamiento quedan auditadas.

In [1]:
import os
import platform
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 100)

def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró el directorio datos/bronze")

ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    "entidad": "costos de almacenaje por centro de distribución",
    "granularidad": "una fila por centro_distribucion y periodo_mes",
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "rutas": {
        "bronze": "datos/bronze/andinalog_warehouse_costs.csv",
        "silver": "datos/silver/andinalog_warehouse_costs_silver.csv",
        "quarantine": "datos/quarantine/andinalog_warehouse_costs_quarantine.csv",
        "informe": "informes/bronze_silver/Informe_B2S_06_Warehouse_Costs.md",
        "notebook": "notebooks/bronze_silver/06_warehouse_costs/B2S_06_Andinalog_Warehouse_Costs.ipynb"
    },
    "columnas_obligatorias": ["centro_distribucion", "rotacion_stock_dias", "perdida_mermas_bob", "periodo_mes", "costo_almacenamiento_mensual_bob"],
    "numericas": ["rotacion_stock_dias", "perdida_mermas_bob", "costo_almacenamiento_mensual_bob"],
    "centros_validos": ["Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"],
    "periodo_formato": r"^\d{4}-(0[1-9]|1[0-2])$",
    "centinelas": [-999],
    "moneda": "BOB según los sufijos de columna observados",
    "imputacion_rotacion": {"habilitada": False, "metodo": "correspondencia_unica_inequivoca", "motivo": "La clave Tarija+2026-08 aparece en dos filas y presenta conflicto de rotación."},
    "criterio_duplicados": "Las filas con la misma clave y valores no idénticos se bloquean; las exactas se resuelven conservando trazabilidad."
}
PATHS = {key: ROOT / value for key, value in CONFIG["rutas"].items()}
for key in ["silver", "quarantine", "informe"]:
    PATHS[key].parent.mkdir(parents=True, exist_ok=True)
print("Raíz detectada:", ROOT)
print("Configuración centralizada para: costos de almacén")
print("Fecha de ejecución UTC:", EXECUTED_AT_UTC)

Raíz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Configuración centralizada para: costos de almacén
Fecha de ejecución UTC: 2026-09-24T22:15:01.580041+00:00


In [2]:
bronze = pd.read_csv(PATHS["bronze"], encoding=CONFIG["lectura"]["encoding"], dtype=CONFIG["lectura"]["dtype"], keep_default_na=CONFIG["lectura"]["keep_default_na"])
bronze.insert(0, "_fila_bronze", range(1, len(bronze) + 1))
perfil = {
    "filas": len(bronze),
    "columnas": bronze.columns.drop("_fila_bronze").tolist(),
    "nulos_o_vacios": {c: int(bronze[c].eq("").sum()) for c in CONFIG["columnas_obligatorias"]},
    "centros_observados": sorted(bronze["centro_distribucion"].unique().tolist()),
    "periodos_observados": sorted(bronze["periodo_mes"].unique().tolist()),
    "duplicados_clave": int(bronze.duplicated(["centro_distribucion", "periodo_mes"], keep=False).sum())
}
print("Perfil Bronze:")
print(perfil)
print(bronze.to_string(index=False))

Perfil Bronze:
{'filas': 6, 'columnas': ['centro_distribucion', 'rotacion_stock_dias', 'perdida_mermas_bob', 'periodo_mes', 'costo_almacenamiento_mensual_bob'], 'nulos_o_vacios': {'centro_distribucion': 0, 'rotacion_stock_dias': 1, 'perdida_mermas_bob': 0, 'periodo_mes': 0, 'costo_almacenamiento_mensual_bob': 0}, 'centros_observados': ['Cochabamba', 'La Paz', 'Oruro', 'Santa Cruz', 'Tarija'], 'periodos_observados': ['2026-08'], 'duplicados_clave': 2}
 _fila_bronze centro_distribucion rotacion_stock_dias perdida_mermas_bob periodo_mes costo_almacenamiento_mensual_bob
            1          Cochabamba               17.23          483014.65     2026-08                           140000
            2              La Paz               16.94           453976.7     2026-08                           180000
            3               Oruro               17.21          506458.32     2026-08                            95000
            4          Santa Cruz               17.73          484945.21 

In [3]:
def preparar(df):
    out = df.copy()
    for col in CONFIG["columnas_obligatorias"]:
        out[col + "_original"] = out[col]
    out["errores_bloqueantes"] = ""
    out["motivos_transformacion"] = ""
    out["motivos_imputacion"] = ""
    out["fue_transformada"] = False
    out["fue_imputada"] = False
    return out

def marcar_error(df, mask, motivo):
    out = df.copy()
    mask = mask.fillna(False)
    current = out.loc[mask, "errores_bloqueantes"]
    out.loc[mask, "errores_bloqueantes"] = np.where(current.eq(""), motivo, current + ";" + motivo)
    return out

def normalizar_campos(df):
    out = df.copy()
    out["centro_distribucion"] = out["centro_distribucion"].str.strip()
    out["periodo_mes"] = out["periodo_mes"].str.strip()
    out["fue_transformada"] = out["fue_transformada"] | out["centro_distribucion"].ne(out["centro_distribucion_original"]) | out["periodo_mes"].ne(out["periodo_mes_original"])
    return out

def convertir_numericas(df):
    out = df.copy()
    for col in CONFIG["numericas"]:
        out[col] = pd.to_numeric(out[col].replace("", pd.NA), errors="coerce")
        conversion_invalida = out[col].isna() & out[col + "_original"].ne("")
        centinela = out[col].isin(CONFIG["centinelas"])
        out[col + "_conversion_invalida"] = conversion_invalida
        out[col + "_centinela_detectado"] = centinela
        out = marcar_error(out, conversion_invalida | centinela, col + ":conversion_invalida_o_centinela")
    return out

def validar_dominio(df):
    out = df.copy()
    centro_invalido = ~out["centro_distribucion"].isin(CONFIG["centros_validos"])
    periodo_invalido = ~out["periodo_mes"].str.match(CONFIG["periodo_formato"], na=False)
    out["centro_distribucion_valido"] = ~centro_invalido
    out["periodo_mes_valido"] = ~periodo_invalido
    out = marcar_error(out, centro_invalido, "centro_distribucion:referencia_invalida")
    out = marcar_error(out, periodo_invalido, "periodo_mes:periodo_invalido")
    return out

def resolver_claves(df):
    out = df.copy()
    clave = ["centro_distribucion", "periodo_mes"]
    out["duplicado_clave"] = out.duplicated(clave, keep=False)
    out["duplicado_exacto"] = out.duplicated(clave + CONFIG["numericas"], keep=False)
    out["conflicto_clave"] = out["duplicado_clave"] & ~out["duplicado_exacto"]
    out = marcar_error(out, out["conflicto_clave"], "duplicado_clave:duplicado_conflictivo")
    out["motivos_transformacion"] = np.where(out["duplicado_exacto"], "duplicado_exacto:conservado_con_trazabilidad", out["motivos_transformacion"])
    return out

def imputar_rotacion(df):
    out = df.copy()
    out["rotacion_stock_dias_imputacion_metodo"] = ""
    out["motivo_imputacion"] = ""
    out["fue_imputada"] = False
    out.loc[out["rotacion_stock_dias"].isna() & out["errores_bloqueantes"].eq(""), "errores_bloqueantes"] = "rotacion_stock_dias:falta_correspondencia_inequivoca"
    out.loc[out["rotacion_stock_dias"].isna(), "calidad_motivo"] = "rotacion_stock_dias:no_imputable"
    return out

def calcular_calidad(df):
    out = df.copy()
    out["calidad_estado"] = out["errores_bloqueantes"].eq("").map({True: "valida", False: "cuarentena"})
    out["calidad_motivo"] = out["calidad_motivo"].where(out["calidad_motivo"].ne(""), out["errores_bloqueantes"].replace("", "sin_incidencias"))
    out["conteo_transformaciones"] = out[["fue_transformada"]].sum(axis=1)
    out["conteo_imputaciones"] = out["fue_imputada"].astype(int)
    return out

import numpy as np
work = bronze.pipe(preparar).pipe(normalizar_campos).pipe(convertir_numericas).pipe(validar_dominio).pipe(resolver_claves).pipe(imputar_rotacion).pipe(calcular_calidad)
silver = work.loc[work["errores_bloqueantes"].eq("")].copy()
quarantine = work.loc[work["errores_bloqueantes"].ne("")].copy()
print("Silver", len(silver), "Cuarentena", len(quarantine))
print(silver[["_fila_bronze", "centro_distribucion", "periodo_mes", "calidad_estado"]].to_string(index=False))
print(quarantine[["_fila_bronze", "centro_distribucion", "periodo_mes", "errores_bloqueantes", "calidad_motivo"]].to_string(index=False))

Silver 4 Cuarentena 2
 _fila_bronze centro_distribucion periodo_mes calidad_estado
            1          Cochabamba     2026-08         valida
            2              La Paz     2026-08         valida
            3               Oruro     2026-08         valida
            4          Santa Cruz     2026-08         valida
 _fila_bronze centro_distribucion periodo_mes                   errores_bloqueantes                   calidad_motivo
            5              Tarija     2026-08 duplicado_clave:duplicado_conflictivo                              NaN
            6              Tarija     2026-08 duplicado_clave:duplicado_conflictivo rotacion_stock_dias:no_imputable


In [4]:
silver.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
assert len(bronze) == len(silver) + len(quarantine)
assert set(silver["_fila_bronze"]).isdisjoint(set(quarantine["_fila_bronze"]))
assert set(silver["_fila_bronze"]) | set(quarantine["_fila_bronze"]) == set(bronze["_fila_bronze"])
assert silver["errores_bloqueantes"].eq("").all()
assert set(silver["centro_distribucion"]).issubset(CONFIG["centros_validos"])
assert silver["periodo_mes"].str.match(CONFIG["periodo_formato"]).all()
assert silver[["centro_distribucion", "periodo_mes"]].duplicated().sum() == 0
print("Controles de conciliación y dominio: OK")
print(pd.DataFrame({"bronze": [len(bronze)], "silver": [len(silver)], "quarantine": [len(quarantine)]}))

Controles de conciliación y dominio: OK
   bronze  silver  quarantine
0       6       4           2


In [5]:
report = [
    "# Informe B2S 06 - AndinaLog Warehouse Costs", "",
    "## Objetivo, entidad y granularidad",
    "Conversión auditada de costos de almacén desde Bronze hacia Silver y cuarentena.",
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad: {CONFIG['granularidad']}.",
    "- Clave funcional: `centro_distribucion` + `periodo_mes`.", "",
    "## Perfil Bronze de esta ejecución",
    f"- Filas Bronze: {perfil['filas']}.",
    f"- Columnas: {perfil['columnas']}.",
    f"- Valores vacíos: {perfil['nulos_o_vacios']}.",
    f"- Centros observados: {perfil['centros_observados']}.",
    f"- Periodos observados: {perfil['periodos_observados']}.",
    f"- Filas con clave duplicada: {perfil['duplicados_clave']}.", "",
    "## Reglas y transformaciones",
    "- El periodo es calendario y se conserva sin desplazamiento de zona horaria.",
    "- Los centros se validan contra el catálogo configurado.",
    "- La moneda observada es BOB por los sufijos de columna.",
    "- Las columnas numéricas conservan original, valor tratado y banderas de conversión/centinela.",
    "- La clave se valida por centro y periodo; los duplicados conflictivos se bloquean.", "",
    "## Imputación y errores",
    "- No se imputa rotación ausente porque Tarija+2026-08 tiene dos filas con estados incompatibles y no existe una correspondencia única inequívoca.",
    "- Ambas filas Tarija+2026-08 permanecen en cuarentena; no se usa cero, promedio, estadística global ni información futura.", "",
    "## Resultado y conciliación",
    f"- Silver: {len(silver)} filas.",
    f"- Cuarentena: {len(quarantine)} filas.",
    f"- Conciliación: Bronze {len(bronze)} = Silver {len(silver)} + cuarentena {len(quarantine)}.",
    "- El plan previo de 5+1 no se mantiene: el perfil exige cuarentena de ambos registros con duplicado conflictivo.", "",
    "## Archivos generados",
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`", "",
    "## Reproducibilidad",
    f"- Fecha UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    "- Bronze se lee como texto sin modificar.",
    "- Ejecutar las celdas en orden; los controles se realizan sobre los CSV persistidos.",
]
PATHS["informe"].write_text("\n".join(report) + "\n", encoding="utf-8")
print("Informe generado:", PATHS["informe"])

Informe generado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_06_Warehouse_Costs.md
